# FahMai Telephone Directory v3
## Agentic RAG — 4-Tool ReAct + /nothink + Context Engineering

**Key changes from v2 (61.4% → target 75%+):**

| Bucket failing in v2 | Root Cause | v3 Fix |
|---|---|---|
| evp_secretary (8 fails) | Agent couldn't find secretaries | grep "SECRETARY" or "EA" + Unit filter |
| vp_identity (6 fails) | VP description search missed | search_by_field on position_en_contains |
| evp_identity_by_description (4) | CTO not searched, only VP | search C-level + VP both |
| dept_member_count (4 fails) | LLM miscounts from truncated CSV | new `count_employees` tool = exact count |
| subsidiary_md/GM (3 fails) | "GM" not in any known field | grep "GENERAL MANAGER" + brand search |
| casual_name_lookup (3 fails) | Ambiguous nickname + dept | search_by_field(nickname_en, dept) |

### Notebook Outline
1. Setup & API  2. Data Loading & EDA  3. Tools (4 tools)
4. Agent Loop   5. System Prompt v3    6. Inference & Submission
7. Local Grade

# 1. Setup

In [1]:
!pip install -q openai pandas tqdm

import os, json, re, time, warnings, io, sys
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from openai import OpenAI

warnings.filterwarnings('ignore')

from kaggle_secrets import UserSecretsClient
TYPHOON_API_KEY = UserSecretsClient().get_secret("TYPHOON_API_KEY")

MODEL      = 'typhoon-v2.5-30b-a3b-instruct'
MAX_TOKENS = 4096

client = OpenAI(api_key=TYPHOON_API_KEY, base_url='https://api.opentyphoon.ai/v1')

try:
    pong = client.chat.completions.create(
        model=MODEL,
        messages=[{'role':'system','content':'/nothink'},
                  {'role':'user','content':'Reply: Ready'}],
        max_tokens=10)
    print("Connection OK:", pong.choices[0].message.content)
except Exception as e:
    print("FAILED:", e)

Connection OK: Ready. How can I assist you today?


# 2. Data Loading & EDA

In [2]:
DATA_DIR   = Path('/kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/')

df_emp = pd.read_csv(DATA_DIR / 'employees.csv').fillna('')
df_questions = pd.read_csv(DATA_DIR / 'questions.csv')

# Clean phone extension
df_emp['Phone Extension'] = df_emp['Phone Extension'].apply(
    lambda x: str(int(float(x))) if x not in ('', None) and str(x).replace('.','').isdigit() else str(x))

N_EMP = len(df_emp)
COLS  = list(df_emp.columns)
print(f"Employees: {N_EMP:,} x {len(COLS)} cols")
print(f"Questions: {len(df_questions):,}")
print(f"Columns: {COLS}")
display(df_emp.head(3))
print(df_emp['Position Level'].value_counts())
print(df_questions['language'].value_counts())

Employees: 1,995 x 19 cols
Questions: 300
Columns: ['Employee ID', 'Department', 'Section', 'Unit', 'Position in Thai', 'Position in English', 'First Name Thai', 'Last Name Thai', 'First Name English', 'Last Name English', 'Nickname Thai', 'Nickname English', 'Email Address', 'Phone Extension', 'Mobile No.', 'Office Location', 'Branch', 'Start Year', 'Position Level']


,Employee ID,Department,Section,Unit,Position in Thai,Position in English,First Name Thai,Last Name Thai,First Name English,Last Name English,Nickname Thai,Nickname English,Email Address,Phone Extension,Mobile No.,Office Location,Branch,Start Year,Position Level
0,2699,CEO,CEO-OFF,CEO,ประธานเจ้าหน้าที่บริหาร,CHIEF EXECUTIVE OFFICER,วชิร,จิรบุญ,VACHIR,CHIRABUN,เบอร์รี่,BERRY,VACHIR.CH@FAHMAI.CO.TH,73048,,FahMai Tower 12F,BKK-R9,2016,C-level
1,8902591,CEO,CEO-OFF,CEO-EA,เลขานุการของ CEO,EXECUTIVE ASSISTANT TO CEO,อรญา,วัชรกาญจน์,ORRAYA,WATCHARAKAN,เป้,PE,ORRAYA.WA@FAHMAI.CO.TH,75665,,FahMai Tower 7F,BKK-R9,2023,Manager
2,6662,CEO,CEO-OFF,CEO-CoS,หัวหน้าสำนักงานประธาน,CHIEF OF STAFF,กิตติคุณ,พงจงรัก,KITTIKHUN,PHONGCHONGRAK,บูม,BOOM,KITTIKHUN.PH@FAHMAI.CO.TH,79367,062-174-6941,FahMai Tower 16F,BKK-R9,2017,VP


Position Level
IC          1573
Lead         188
Manager      140
Director      63
VP            24
C-level        7
Name: count, dtype: int64
language
th    234
en     66
Name: count, dtype: int64


# 3. Tools (4 tools)
- `grep_csv`: broad substring search all columns
- `search_by_field`: precise column-level filter
- `count_employees`: exact count (fixes dept_member_count bucket)
- `list_employees`: list all matching as compact rows

In [3]:
# ─── Tool 1: grep_csv ────────────────────────────────────────────────────────
def grep_csv(pattern: str, max_results: int = 15) -> str:
    p = (pattern or '').strip()
    if not p:
        return json.dumps({'error': 'empty'})
    mask = df_emp.astype(str).apply(
        lambda col: col.str.contains(re.escape(p), case=False, na=False)
    ).any(axis=1)
    hits = df_emp[mask]
    total = len(hits)
    if total == 0:
        return json.dumps({'total': 0, 'note': f'No match for "{p}"'})
    return json.dumps({
        'total': total,
        'returned': min(total, max_results),
        'truncated': total > max_results,
        'csv': hits.head(max_results).to_csv(index=False)
    }, ensure_ascii=False)

# ─── Tool 2: search_by_field ─────────────────────────────────────────────────
def search_by_field(
    department='', position_level='', unit='', section='',
    position_en_contains='', position_th_contains='',
    nickname_th='', nickname_en='',
    first_name_th='', last_name_th='',
    first_name_en='', last_name_en='',
    branch='', max_results=15
) -> str:
    df = df_emp.copy()
    def filt(col, val):
        if val:
            return df[col].str.contains(re.escape(val.strip()), case=False, na=False)
        return pd.Series([True]*len(df), index=df.index)
    mask = (
        filt('Department', department) &
        filt('Position Level', position_level) &
        filt('Unit', unit) &
        filt('Section', section) &
        filt('Position in English', position_en_contains) &
        filt('Position in Thai', position_th_contains) &
        filt('Nickname Thai', nickname_th) &
        filt('Nickname English', nickname_en) &
        filt('First Name Thai', first_name_th) &
        filt('Last Name Thai', last_name_th) &
        filt('First Name English', first_name_en) &
        filt('Last Name English', last_name_en) &
        filt('Branch', branch)
    )
    hits = df[mask]
    total = len(hits)
    if total == 0:
        return json.dumps({'total': 0, 'note': 'No match'})
    return json.dumps({
        'total': total,
        'returned': min(total, max_results),
        'csv': hits.head(max_results).to_csv(index=False)
    }, ensure_ascii=False)

# ─── Tool 3: count_employees ─────────────────────────────────────────────────
def count_employees(
    department='', position_level='', unit='', section='',
    position_en_contains='', nickname_en='', nickname_th='',
    branch=''
) -> str:
    'Return EXACT count matching given filters. Use for count/how-many questions.'
    df = df_emp.copy()
    def filt(col, val):
        if val:
            return df[col].str.contains(re.escape(val.strip()), case=False, na=False)
        return pd.Series([True]*len(df), index=df.index)
    mask = (
        filt('Department', department) &
        filt('Position Level', position_level) &
        filt('Unit', unit) &
        filt('Section', section) &
        filt('Position in English', position_en_contains) &
        filt('Nickname English', nickname_en) &
        filt('Nickname Thai', nickname_th) &
        filt('Branch', branch)
    )
    count = int(df[mask].shape[0])
    return json.dumps({'exact_count': count, 'filters_used': {
        'department': department, 'position_level': position_level,
        'unit': unit, 'section': section,
        'position_en_contains': position_en_contains
    }})

# ─── Tool schemas ─────────────────────────────────────────────────────────────
GREP_CSV_TOOL = {
    'type': 'function',
    'function': {
        'name': 'grep_csv',
        'description': (
            f'Search FahMai employee directory ({N_EMP} rows) by substring across ALL columns. '
            'Columns: ' + ', '.join(COLS) + '. '
            'Use for: name, nickname, email, phone, department, position keywords. '
            'Returns total count + CSV rows.'
        ),
        'parameters': {'type':'object','properties':{
            'pattern':{'type':'string','description':'Search keyword. E.g. "วชิร","BERRY","73048","SECRETARY","GENERAL MANAGER"'},
            'max_results':{'type':'integer','default':15}
        },'required':['pattern']},
    }
}

SEARCH_BY_FIELD_TOOL = {
    'type': 'function',
    'function': {
        'name': 'search_by_field',
        'description': (
            'Filter employees by SPECIFIC columns (more precise than grep_csv). '
            'All parameters optional, case-insensitive substring match. '
            'KEY PATTERNS: '
            '(1) Dept+VP code: department="RET" + position_level="VP" for RETVP. '
            '(2) Secretary: position_en_contains="SECRETARY" + unit="WKVP" for WKVP secretary. '
            '(3) C-suite: position_en_contains="CHIEF OPERATING" for COO. '
            '(4) Nickname+dept: nickname_en="TIK" + department="RET". '
            'position_level values: C-level, VP, Director, Manager, Lead, IC.'
        ),
        'parameters': {'type':'object','properties':{
            'department':{'type':'string'},
            'position_level':{'type':'string','description':'"VP","C-level","Director","Manager","Lead","IC"'},
            'unit':{'type':'string','description':'Unit code e.g. "RETVP","CEO","WKVP","CHRO"'},
            'section':{'type':'string'},
            'position_en_contains':{'type':'string','description':'e.g. "CHIEF OPERATING","SECRETARY","GENERAL MANAGER","VICE PRESIDENT FLEET"'},
            'position_th_contains':{'type':'string'},
            'nickname_th':{'type':'string'},
            'nickname_en':{'type':'string'},
            'first_name_th':{'type':'string'},
            'last_name_th':{'type':'string'},
            'first_name_en':{'type':'string'},
            'last_name_en':{'type':'string'},
            'branch':{'type':'string'},
            'max_results':{'type':'integer','default':15}
        }},
    }
}

COUNT_TOOL = {
    'type': 'function',
    'function': {
        'name': 'count_employees',
        'description': (
            'Get EXACT employee count matching filters. '
            'ALWAYS use this for "how many", "กี่คน", "จำนวน" questions. '
            'Returns {"exact_count": N}. '
            'Examples: count_employees(department="MKT") -> 105, '
            'count_employees(unit="B2B-SUP") -> 16.'
        ),
        'parameters': {'type':'object','properties':{
            'department':{'type':'string'},
            'position_level':{'type':'string'},
            'unit':{'type':'string','description':'Exact unit code e.g. "MKT-BRND","B2B-SUP","KS-PD"'},
            'section':{'type':'string'},
            'position_en_contains':{'type':'string'},
            'nickname_en':{'type':'string'},
            'nickname_th':{'type':'string'},
            'branch':{'type':'string'}
        }},
    }
}

TOOLS = [GREP_CSV_TOOL, SEARCH_BY_FIELD_TOOL, COUNT_TOOL]
TOOL_MAP = {
    'grep_csv': grep_csv,
    'search_by_field': search_by_field,
    'count_employees': count_employees,
}

# Smoke tests
print("grep_csv CEO:", json.loads(grep_csv('CEO',max_results=3))['total'])
print("search_by_field RET+VP:", json.loads(search_by_field(department='RET',position_level='VP'))['total'])
print("search_by_field SECRETARY+WKVP:", json.loads(search_by_field(position_en_contains='SECRETARY',unit='WKVP'))['total'])
print("count MKT:", json.loads(count_employees(department='MKT'))['exact_count'])

grep_csv CEO: 10
search_by_field RET+VP: 3
search_by_field SECRETARY+WKVP: 1
count MKT: 105


# 4. ReAct Agent Loop

In [4]:
def run_agent(messages, tools, tool_map, max_iters=6, verbose=False):
    'ReAct loop: LLM -> tool_calls -> results -> repeat until answer'
    for i in range(max_iters):
        try:
            resp = client.chat.completions.create(
                model=MODEL, messages=messages, tools=tools,
                max_tokens=MAX_TOKENS, temperature=0.0)
        except Exception as e:
            time.sleep(3)
            try:
                resp = client.chat.completions.create(
                    model=MODEL, messages=messages, tools=tools,
                    max_tokens=MAX_TOKENS, temperature=0.0)
            except Exception as e2:
                raise RuntimeError(f"API error: {e2}")

        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            return (msg.content or '').strip()

        if verbose:
            for c in msg.tool_calls:
                print(f"  [iter {i}] {c.function.name}({c.function.arguments[:60]})")

        for call in msg.tool_calls:
            fn = call.function.name
            try:
                args = json.loads(call.function.arguments)
            except Exception:
                args = {}
            result = str(tool_map[fn](**args)) if fn in tool_map else f"Unknown: {fn}"
            messages.append({'role':'tool','tool_call_id':call.id,'content':result})

    return '[max_iters]'

print("Agent loop ready")

Agent loop ready


# 5. System Prompt v3
Key improvements:
- `/nothink` (Qwen3 MoE no-thinking mode)
- Explicit Unit code decoding table
- Secretary lookup pattern
- GM/GENERAL MANAGER lookup pattern
- Count questions → always use count_employees
- C-suite disambiguation
- Language mirroring strict rule

In [5]:
SP_LINES = [
    "/nothink",
    "",
    "You are the FahMai Company Telephone Directory Assistant.",
    "ALWAYS use tools to look up data. NEVER guess or fabricate.",
    "",
    "== TOOL USAGE GUIDE ==",
    "",
    "STEP 1 — Decode position codes (RETVP, LOGFL, etc.):",
    "  Pattern: [DEPT][LEVEL] where DEPT=2-3 chars, LEVEL=VP/EVP/FL/DIR etc.",
    "  Examples: RETVP -> department=RET + position_level=VP",
    "            LOGFL -> department=LOG + position_level=Lead",
    "            WKVP  -> department=WK  + position_level=VP  (also unit=WKVP)",
    "            SFVP  -> department=SF  + position_level=VP",
    "            DNVP  -> department=DN  + position_level=VP",
    "  Use: search_by_field(department='RET', position_level='VP')",
    "",
    "STEP 2 — C-suite titles (CEO/CFO/COO/CTO/CHRO/CPO/CMO):",
    "  Use: search_by_field(position_en_contains='CHIEF EXECUTIVE') for CEO",
    "       search_by_field(position_en_contains='CHIEF OPERATING') for COO",
    "       search_by_field(position_en_contains='CHIEF TECHNOLOGY') for CTO",
    "       search_by_field(position_en_contains='CHIEF FINANCIAL') for CFO",
    "       search_by_field(position_en_contains='CHIEF HUMAN RESOURCES') for CHRO",
    "       search_by_field(position_en_contains='CHIEF PRODUCT') for CPO",
    "",
    "STEP 3 — Secretary/assistant of a VP/C-level:",
    "  Pattern: 'เลขาของ RETVP' or 'secretary of RETVP'",
    "  Use: search_by_field(position_en_contains='SECRETARY', unit='RETVP')",
    "  Or: grep_csv('SECRETARY OF RETVP')",
    "  Or: search_by_field(unit='RETVP') to see all unit members",
    "",
    "STEP 4 — General Manager of subsidiary brand:",
    "  Brands: สายฟ้า(SF/Saifah), ดาวเหนือ(DN/Daonuea), วงโคจร(WK/Wongkhojon),",
    "          จุดเชื่อม(JC/Judchuem), คลื่นเสียง(KS/Kluensiang)",
    "  Use: grep_csv('GENERAL MANAGER') to find all GMs",
    "       then grep_csv('SAIFAH') or grep_csv('สายฟ้า') to narrow down",
    "  Or: search_by_field(position_en_contains='GENERAL MANAGER', department='SF')",
    "",
    "STEP 5 — Count questions ('กี่คน', 'how many', 'จำนวน'):",
    "  ALWAYS use count_employees tool. NEVER count manually.",
    "  For 'MKT กี่คน': count_employees(department='MKT')",
    "  For 'B2B-SUP กี่คน': count_employees(unit='B2B-SUP') OR count_employees(section='B2B-SUP')",
    "",
    "STEP 6 — VP by description ('ดูแล tech', 'in charge of operations'):",
    "  'tech' -> search CTO first: search_by_field(position_en_contains='CHIEF TECHNOLOGY')",
    "  'operations' -> search COO: search_by_field(position_en_contains='CHIEF OPERATING')",
    "  'finance' -> search CFO: search_by_field(position_en_contains='CHIEF FINANCIAL')",
    "  'HR/human resources' -> search_by_field(position_en_contains='CHIEF HUMAN RESOURCES')",
    "  Then also search VP level: search_by_field(position_en_contains='VICE PRESIDENT TECHNOLOGY')",
    "",
    "STEP 7 — Nickname search:",
    "  Use search_by_field(nickname_en='NUTTY') or grep_csv('NUTTY')",
    "  If 0 results, respond 'no nickname on record' (EN) or 'ไม่มีชื่อเล่นในระบบ' (TH)",
    "",
    "== LANGUAGE RULE ==",
    "Thai question -> ALL Thai answer.",
    "English question -> ALL English answer.",
    "Never mix languages in the same answer.",
    "",
    "== REFUSAL RULES (EXACT PHRASES ONLY) ==",
    "Refuse ONLY if asking for: salary/bonus/age/religion/performance/",
    "promotion decision/disciplinary/personal opinions/non-FahMai company data.",
    "",
    "Thai refusals:",
    "  - General forbidden: 'ไม่สามารถให้ข้อมูลนี้ได้'",
    "  - Other company: 'ไม่ใช่ข้อมูลของฟ้าใหม่'",
    "  - Opinion: 'ไม่สามารถให้ความเห็นได้'",
    "  - No nickname in DB: 'ไม่มีชื่อเล่นในระบบ'",
    "  - Not found: 'ไม่พบข้อมูล'",
    "",
    "English refusals:",
    "  - General forbidden: 'cannot provide this information'",
    "  - Other company: 'not Fahmai data'",
    "  - Opinion: 'cannot offer an opinion'",
    "  - No nickname in DB: 'no nickname on record'",
    "  - Not found: 'no record found'",
    "",
    "== LEAKAGE RULE ==",
    "On refusal: output ONLY the phrase above.",
    "Do NOT include Employee ID, Phone Extension, or email in refusal.",
    "",
    "== ANSWER STYLE ==",
    "Concise. Include Thai + English name when available.",
    "For phone queries: state the extension number.",
    "For count: state the EXACT number from count_employees tool.",
    "For family/surname questions: answer YES with examples from tool data.",
]

SYSTEM_PROMPT_V3 = "\n".join(SP_LINES)
print("System Prompt v3 ready:", len(SYSTEM_PROMPT_V3), "chars")
print("Starts with /nothink:", SYSTEM_PROMPT_V3.startswith('/nothink'))

System Prompt v3 ready: 3870 chars
Starts with /nothink: True


# 6. Inference & Submission

In [6]:
def process_question(q_id, question, language, verbose=False):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT_V3},
        {'role': 'user',   'content': question},
    ]
    fallback = 'ไม่พบข้อมูล' if language == 'th' else 'no record found'
    try:
        ans = run_agent(messages, TOOLS, TOOL_MAP, max_iters=6, verbose=verbose)
        if not ans or ans.strip() in ('', '[max_iters]'):
            return fallback
        return ans.strip()
    except Exception as e:
        print(f"  ERROR {q_id}: {e}")
        return fallback

results = []
print(f"Inference on {len(df_questions)} questions | Model: {MODEL}")
print(f"Tools: {[t['function']['name'] for t in TOOLS]}")

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions), desc='Answering'):
    q_id     = row['id']
    question = row['question']
    language = row['language'] if 'language' in row.index else 'th'
    answer   = process_question(q_id, question, language, verbose=False)
    results.append({'id': q_id, 'response': answer})

submission_df = pd.DataFrame(results)
submission_df.to_csv('submission.csv', index=False, encoding='utf-8')
print(f"Saved {len(submission_df)} rows -> submission.csv")
display(submission_df.head(10))

Inference on 300 questions | Model: typhoon-v2.5-30b-a3b-instruct
Tools: ['grep_csv', 'search_by_field', 'count_employees']


Answering:   0%|          | 0/300 [00:00<?, ?it/s]

Saved 300 rows -> submission.csv


,id,response
0,g001,The RETVP is Wiriya Chanchai (ติ๊ก). \nEmploy...
1,g002,- **คึกฤทธิ์ บุษราคัมวงศ์** (KUKRIT BUSARAKHAM...
2,g004,LEGVP คือ ไพโรจน์ มหากุล (Phairoj Mahakun)
3,g005,พงษ์กานต์ ราชชากัญญ์ (PONGKAN RAJCHAKAN) เป็น ...
4,g007,ปัจจุบัน LOGFL มีทั้งหมด 15 คน ดังนี้:\n\n1. *...
5,g008,เรืองศักดิ์ เทพเกียรติกำจร (RUANGSAK THEPKIATK...
6,g009,SUPVP มี 2 คน:\n\n1. **ดาริกา อาวุทธ์ดี** (DAR...
7,g011,The CHRO is Nathamon Aphichaidi (ณฐามน อภิชัยด...
8,g012,"- **คะวัง กอบสุขรัตน์** (KWANG KOBSOOKRAT), MK..."
9,g014,LOGVP ปัจจุบันมี 2 คน:\n\n1. **ณัฐกานต์ ศรีอาร...


# 7. Local Validation

In [7]:
import subprocess

grade_script = DATA_DIR / 'grade.py'
train_labels = DATA_DIR / 'train_labels.json'

if grade_script.exists() and train_labels.exists():
    res = subprocess.run(
        ['python', str(grade_script), 'submission.csv', str(train_labels)],
        capture_output=True, text=True, encoding='utf-8')
    print("="*60)
    print(res.stdout)
    if res.stderr:
        print("STDERR:", res.stderr[:300])
else:
    print("grade.py not found in", DATA_DIR)

Scored 158 items against /kaggle/input/competitions/super-ai-engineer-season-6-fahmai-telephone-directory/train_labels.json
Passed: 93/158 = 58.9%

Bucket                             pass/total    rate
--------------------------------------------------------
nickname_grid                    5/      17   29.4%
refuse                           6/      15   40.0%
evp_secretary                    7/       9   77.8%
vp_identity                      7/       9   77.8%
casual_name_lookup               3/       9   33.3%
evp_identity_by_code             8/       8  100.0%
evp_identity_by_description      5/       8   62.5%
name_lookup                      2/       8   25.0%
dept_listing_medium              8/       8  100.0%
dept_member_count                5/       7   71.4%
dept_listing_small               2/       6   33.3%
extension_reverse                5/       5  100.0%
hard_nickname_variant            2/       5   40.0%
evp_vs_vp_disambig               3/       4   75.0%
email_mobile_

### Quick analysis


In [8]:
submission_df['len'] = submission_df['response'].str.len()
print(submission_df['len'].describe())
print()
print("Shortest responses (check for wrong refusals):")
display(submission_df.nsmallest(8,'len')[['id','response']])

count     300.000000
mean      236.373333
std       413.505037
min         9.000000
25%        23.000000
50%        89.000000
75%       179.000000
max      2297.000000
Name: len, dtype: float64

Shortest responses (check for wrong refusals):


,id,response
173,g230,21 people
11,g018,ไม่พบข้อมูล
31,g046,ไม่พบข้อมูล
36,g052,ไม่พบข้อมูล
39,g058,ไม่พบข้อมูล
43,g063,ไม่พบข้อมูล
51,g073,ไม่พบข้อมูล
53,g075,ไม่พบข้อมูล
